# Raster Validation

Validates raster built-up datasets (TEMPO, GHSL Built-S/V/H, Google OBT) against reference building footprints.

Pipeline per city:
1. Tile the AOI into 1 km × 1 km cells
2. For each enabled raster candidate, read the year-specific file (`{city_slug}_{name}_{year}.tif`)
3. Rasterize reference footprints onto the candidate grid (fractional coverage via oversampling)
4. Compute tile-level binary (TP/FP/FN/F1) and area-based metrics
5. Save tile metrics, city summary, and figures to `outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
os.chdir("/content/drive/MyDrive/urban_validation/")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/urban_validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose raster outputs already exist.
# When False (default), cities with an existing sentinel parquet are skipped automatically.
OVERWRITE = True

In [ ]:
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)   # clones repo -> sys.path; cwd stays on Drive

In [ ]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

raster_datasets = _cfg_preview.get("raster", {}).get("datasets", [])
enabled = [
    f"{d['name']}" + (f"_{d['year']}" if d.get("year") is not None else "")
    for d in raster_datasets
    if d.get("enabled", True)
]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled raster datasets: {enabled}")

In [ ]:
# Patch overwrite flag and instantiate the Validator.
# This reads the config and AOI tracker, resolves file paths,
# and logs how many city datasets are queued.
import tempfile

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
# Patch root_dir to the actual PROJECT_ROOT on this Colab instance.
# The YAML may contain a stale path if the project was moved on Drive.
_cfg_patched["root_dir"] = str(PROJECT_ROOT)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nCities queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

In [ ]:
# Preview: show which raster files will be looked up for each city × dataset combination.
# Files that don't exist on disk are flagged so you can fix paths before running.
import pandas as pd
from pathlib import Path

data_dir = Path(_cfg_patched.get("root_dir", str(PROJECT_ROOT))) / _cfg_patched.get("data_dir", "data/01_raw")

preview_rows = []
for ds in v.datasets:
    city_slug    = ds["id"].lower()
    city_slug_us = city_slug.replace("-", "_")
    rast_dir     = data_dir / ds["id"] / "raster"

    for cand in _cfg_patched.get("raster", {}).get("datasets", []):
        if not cand.get("enabled", True):
            continue
        ds_name = cand["name"].replace("-", "_")
        year    = cand.get("year")
        if year is not None:
            fpath = rast_dir / f"{city_slug_us}_{ds_name}_{year}.tif"
        else:
            matches = sorted(rast_dir.glob(f"{city_slug_us}_{ds_name}*.tif"))
            fpath   = matches[0] if matches else rast_dir / f"{city_slug_us}_{ds_name}_?.tif"
        preview_rows.append({
            "city":    ds["id"],
            "dataset": f"{ds_name}_{year}" if year is not None else ds_name,
            "file":    fpath.name,
            "exists":  fpath.exists(),
        })

preview_df = pd.DataFrame(preview_rows)
missing = preview_df[~preview_df["exists"]]
print(f"Total city × dataset pairs: {len(preview_df)}")
print(f"Missing files:              {len(missing)}")
display(preview_df)

In [ ]:
results = v.validate_raster()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"city": k, "status": "ok" if ok else "failed"} for k, ok in results.items()]
)
print(f"\nDone — {len(summary)} cities processed.\n")
display(summary.groupby("status")["city"].count().rename("count").to_frame())
display(summary)

## Post-run diagnostics

### Per-pixel MAE and RMSE by dataset

MAE (mean absolute error) and RMSE are computed per valid pixel within each tile, then averaged city-wide.
Unlike the signed area bias (which allows over- and under-prediction to cancel), per-pixel MAE/RMSE
measures the average per-pixel discrepancy in built area. A large RMSE/MAE gap indicates error is
concentrated in a small number of tiles rather than evenly spread.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

metrics_root = PROJECT_ROOT / "outputs" / "metrics"

rows = []
for city_dir in sorted(metrics_root.iterdir()):
    if not city_dir.is_dir():
        continue
    for p in city_dir.glob("raster_metrics_tiles_*.parquet"):
        # Extract dataset name from filename: raster_metrics_tiles_{dataset}_{grid}.parquet
        stem = p.stem.replace("raster_metrics_tiles_", "")
        parts = stem.rsplit("_", 1)
        ds_name = parts[0] if len(parts) == 2 else stem
        grid = parts[1] if len(parts) == 2 else "unknown"
        df = pd.read_parquet(p)
        if "mae_area_m2" not in df.columns or "rmse_area_m2" not in df.columns:
            continue
        valid_tiles = df[df["n_valid"] > 0]
        rows.append({
            "city": city_dir.name,
            "dataset": ds_name,
            "grid": grid,
            "n_tiles": len(valid_tiles),
            "f1_mean": valid_tiles["f1"].mean(),
            "mae_area_m2_mean": valid_tiles["mae_area_m2"].mean(),
            "rmse_area_m2_mean": valid_tiles["rmse_area_m2"].mean(),
            "rmse_mae_ratio": valid_tiles["rmse_area_m2"].mean() / valid_tiles["mae_area_m2"].mean()
            if valid_tiles["mae_area_m2"].mean() > 0 else np.nan,
            "signed_bias_mean": valid_tiles["rel_area_error"].mean(),
        })

mae_rmse_df = pd.DataFrame(rows)
if not mae_rmse_df.empty:
    summary = (
        mae_rmse_df.groupby(["dataset", "grid"])[
            ["f1_mean", "mae_area_m2_mean", "rmse_area_m2_mean", "rmse_mae_ratio", "signed_bias_mean"]
        ]
        .mean()
        .round(4)
    )
    print("=== MAE / RMSE summary (mean across cities, per eval grid) ===")
    display(summary)
else:
    print("No raster tile metric parquets found — run validate_raster() first.")

### Zero-reference tile diagnostic

Tiles with valid pixels but zero reference built area (ref_area_m2 = 0) are unexpected: every city
was selected because it has reference buildings. A high count here likely indicates a spatial mismatch
between the raster tile and the reference dataset (wrong CRS, path issue, or reference file gap).

The table below shows, per dataset and city, the count and share of such tiles. Datasets are flagged
if more than 5 % of their valid tiles are zero-reference.

In [ ]:
zero_ref_rows = []
for city_dir in sorted(metrics_root.iterdir()):
    if not city_dir.is_dir():
        continue
    for p in city_dir.glob("raster_metrics_tiles_*.parquet"):
        stem = p.stem.replace("raster_metrics_tiles_", "")
        parts = stem.rsplit("_", 1)
        ds_name = parts[0] if len(parts) == 2 else stem
        grid = parts[1] if len(parts) == 2 else "unknown"
        df = pd.read_parquet(p)
        valid_tiles = df[df["n_valid"] > 0]
        if valid_tiles.empty:
            continue
        zero_ref = valid_tiles[valid_tiles["ref_area_m2"].fillna(0) == 0]
        n_total = len(valid_tiles)
        n_zero = len(zero_ref)
        zero_ref_rows.append({
            "city": city_dir.name,
            "dataset": ds_name,
            "grid": grid,
            "n_valid_tiles": n_total,
            "n_zero_ref_tiles": n_zero,
            "pct_zero_ref": round(100.0 * n_zero / n_total, 1) if n_total > 0 else 0.0,
            "flagged": n_zero / n_total > 0.05 if n_total > 0 else False,
        })

zero_ref_df = pd.DataFrame(zero_ref_rows)
if not zero_ref_df.empty:
    nonzero = zero_ref_df[zero_ref_df["n_zero_ref_tiles"] > 0].sort_values(
        ["pct_zero_ref", "city"], ascending=[False, True]
    )
    flagged = zero_ref_df[zero_ref_df["flagged"]]

    print(f"Cities × datasets with at least one zero-reference tile: {len(nonzero)}")
    print(f"Flagged (>5 % zero-reference tiles):                      {len(flagged)}\n")

    if not flagged.empty:
        print("=== FLAGGED datasets (investigate spatial/path alignment) ===")
        display(
            flagged[["city", "dataset", "grid", "n_valid_tiles", "n_zero_ref_tiles", "pct_zero_ref"]]
            .reset_index(drop=True)
        )
        print()

    print("=== All city × dataset pairs with zero-reference tiles ===")
    display(
        nonzero[["city", "dataset", "grid", "n_valid_tiles", "n_zero_ref_tiles", "pct_zero_ref", "flagged"]]
        .reset_index(drop=True)
    )
else:
    print("No raster tile metric parquets found — run validate_raster() first.")